In [ ]:
import pandas as pd
import numpy as np
import wandb

api = wandb.Api()

ENTITY  = "X"
PROJECT = "X"

runs = list(api.runs(f"{ENTITY}/{PROJECT}"))
print(f"Loaded {len(runs)} runs")

Loaded 197 runs


In [14]:
filters = {
    "config.dataset": "jigsaw",
    "config.strategy": {"$in": ["bo", "disagreement"]},
    "$or": [
        {"config.batch_size": {"$in": [512]}},
        {"config.batch_size": {"$in": ["512"]}},
    ],
    "$or": [
        {"config.epochs_opt": {"$in": [10]}},
        {"config.epochs_opt": {"$in": ["10"]}},
    ],
}
runs = api.runs(f"{ENTITY}/{PROJECT}", filters=filters)


In [8]:
MIN_T_END = 1000

finished_runs = []
for run in runs:
    if run.state != "finished":
        continue

    h = pd.DataFrame(run.scan_history(keys=["audit/outer/T_size", "_step"]))
    if h.empty:
        continue

    h["audit/outer/T_size"] = pd.to_numeric(h["audit/outer/T_size"], errors="coerce")
    t_end = h["audit/outer/T_size"].dropna().max()   # robust: max observed T_size

    if pd.notna(t_end) and t_end >= MIN_T_END:
        finished_runs.append(run)

runs = finished_runs
print("Kept runs:", len(runs))


Kept runs: 20


In [ ]:
import pandas as pd

OUTER_KEYS = {
    "T_size": "audit/outer/T_size",
    "err_active": "audit/outer/err_mid",
    "width_abs": "audit/outer/width",
    "iter": "audit/outer/iter",

    "delta_est": "audit/outer/delta_mid",

    # keep bounds in history (these are per-iter)
    "h_min": "audit/cerm/delta_min_true",
    "h_max": "audit/cerm/delta_max_true",
}

HISTORY_KEYS = ["_step"] + list(OUTER_KEYS.values())

ts_rows = []
run_rows = []  # per-run scalars

for run in runs:
    h = pd.DataFrame(run.scan_history(keys=HISTORY_KEYS))
    if h.empty:
        continue

    # rename to short names
    h = h.rename(columns={v: k for k, v in OUTER_KEYS.items()})

    # add identifiers
    h["run_id"] = run.id
    h["seed"] = run.config.get("seed")
    h["strategy"] = run.config.get("strategy")
    h["surrogate_batch_size"] = run.config.get("surrogate_batch_size")
    h["epochs_opt"] = run.config.get("epochs_opt")

    # numeric coercion (include h_min/h_max)
    for c in ["T_size", "err_active", "width_abs", "iter", "h_min", "h_max", "_step"]:
        if c in h.columns:
            h[c] = pd.to_numeric(h[c], errors="coerce")

    # keep only valid iteration rows
    h = h.dropna(subset=["iter"])

    ts_rows.append(h)

    # --- per-run scalars from summary (broadcast later) ---
    run_rows.append({
        "run_id": run.id,
        "delta_bb_final": run.summary.get("audit/final/delta_bb"),
        "delta_bb_init":  run.summary.get("audit/init/delta_bb"),
    })

ts = pd.concat(ts_rows, ignore_index=True)

run_df = pd.DataFrame(run_rows)
ts = ts.merge(run_df, on="run_id", how="left")


In [22]:
ts

,_step,T_size,err_active,width_abs,iter,delta_est,run_id,seed,strategy,surrogate_batch_size,epochs_opt,delta_bb_final,delta_bb_init
0,1,32,0.187538,1.708654,1,-0.049506,hn7pwkya,60,disagreement,16,10,NaN,0.138032
1,2,48,0.223576,1.640904,2,-0.085544,hn7pwkya,60,disagreement,16,10,NaN,0.138032
2,3,64,0.187503,1.703217,3,-0.049471,hn7pwkya,60,disagreement,16,10,NaN,0.138032
3,4,80,0.208136,1.657148,4,-0.070104,hn7pwkya,60,disagreement,16,10,NaN,0.138032
4,1,64,0.152676,1.522568,1,-0.014627,a1g5mgb6,60,disagreement,16,10,NaN,0.138049
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2469,34,532,0.008175,0.119529,34,0.145485,fwdp6ztw,89,disagreement,16,10,NaN,0.153660
2470,35,548,0.013705,0.109782,35,0.139955,fwdp6ztw,89,disagreement,16,10,NaN,0.153660
2471,36,564,0.002227,0.104626,36,0.151433,fwdp6ztw,89,disagreement,16,10,NaN,0.153660
2472,37,580,0.011915,0.134026,37,0.141745,fwdp6ztw,89,disagreement,16,10,NaN,0.153660


In [23]:
ts.to_csv("eval_BAFA_bios_jigsaw.csv", index=False)

In [ ]:
import pandas as pd

def _unwrap(v):
    # W&B sometimes stores config values as {"value": x}
    if isinstance(v, dict) and "value" in v and len(v) == 1:
        return v["value"]
    return v

def cfg(run, key, default=None):
    v = run.config.get(key, default)
    return _unwrap(v)

def wandb_env_info(run):
    wb = _unwrap(run.config.get("_wandb", {})) or {}
    # Your pasted structure: _wandb["e"] is dict keyed by writerId
    e = wb.get("e", {})
    if isinstance(e, dict) and len(e) > 0:
        env = next(iter(e.values()))
    else:
        env = {}

    gpu_name = None
    gpu_mem = None
    cuda_ver = env.get("cudaVersion")
    py_ver = env.get("python")
    cpu_count = env.get("cpu_count")
    cpu_count_logical = env.get("cpu_count_logical")

    gpu_list = env.get("gpu_nvidia") or []
    if gpu_list:
        gpu_name = gpu_list[0].get("name")
        gpu_mem = gpu_list[0].get("memoryTotal")  # string in your dump

    return {
        "gpu": gpu_name,
        "gpu_mem_total": gpu_mem,
        "cuda_version": cuda_ver,
        "python_version": py_ver,
        "cpu_count": cpu_count,
        "cpu_count_logical": cpu_count_logical,
        "git_commit": (env.get("git") or {}).get("commit"),
    }

# --- Build run-level meta table (appendix-friendly) ---
meta_rows = []
for run in runs:
    # W&B runtime is usually stored in seconds in summary under "_runtime"
    runtime_s = run.summary.get("_runtime", None)

    env = wandb_env_info(run)

    meta_rows.append({
        "run_id": run.id,
        "name": getattr(run, "name", None),
        "state": getattr(run, "state", None),

        "dataset": cfg(run, "dataset"),
        "blackbox": cfg(run, "blackbox"),
        "strategy": cfg(run, "strategy"),
        "seed": cfg(run, "seed"),

        # key hyperparams (pick what you want in appendix)
        "size_T": cfg(run, "size_T"),
        "iterations": cfg(run, "iterations"),
        "k_batch": cfg(run, "k_batch"),
        "candidate_pool_M": cfg(run, "candidate_pool_M"),
        "batch_size": cfg(run, "batch_size"),
        "epochs_sur": cfg(run, "epochs_sur"),
        "epochs_opt": cfg(run, "epochs_opt"),
        "surrogate_lr": cfg(run, "surrogate_lr"),
        "lambda_penalty": cfg(run, "lambda_penalty"),
        "reg_alpha": cfg(run, "reg_alpha"),

        "runtime_sec": runtime_s,

        **env,
    })

meta_df = pd.DataFrame(meta_rows)

# optional: convert runtime to hh:mm:ss for appendix readability
meta_df["runtime_hms"] = pd.to_timedelta(meta_df["runtime_sec"], unit="s").astype(str)

meta_df.head()


,run_id,name,state,dataset,blackbox,strategy,seed,size_T,iterations,k_batch,...,reg_alpha,runtime_sec,gpu,gpu_mem_total,cuda_version,python_version,cpu_count,cpu_count_logical,git_commit,runtime_hms
0,8oiklpdh,main_strategies-seed0,failed,None,None,None,0,64,500,NaN,...,NaN,42037.482061,None,None,None,None,None,None,None,0 days 11:40:37.482060765
1,yvgcs6h0,main_strategies_reg-seed0,finished,None,None,None,0,64,50,NaN,...,NaN,22435.576297,None,None,None,None,None,None,None,0 days 06:13:55.576297234
2,0hfwuf3i,main_strategies_reg-seed1,crashed,None,None,None,1,64,50,NaN,...,NaN,8431.141717,None,None,None,None,None,None,None,0 days 02:20:31.141717001
3,c5d96f5u,main_strategies_reg-seed3,finished,None,None,None,3,64,50,NaN,...,NaN,33081.915741,None,None,None,None,None,None,None,0 days 09:11:21.915741414
4,3ivf6u5h,main_strategies_reg-seed4,finished,None,None,None,4,64,50,NaN,...,NaN,27638.007117,None,None,None,None,None,None,None,0 days 07:40:38.007116683


In [ ]:
# aggregate across seeds per dataset/blackbox/strategy
agg = (meta_df
       .dropna(subset=["runtime_sec"])
       .groupby(["dataset", "blackbox", "strategy"])
       .agg(
           n_runs=("run_id", "count"),
           runtime_mean_sec=("runtime_sec", "mean"),
           runtime_std_sec=("runtime_sec", "std"),
           batch_size=("batch_size", "first"),
           epochs_opt=("epochs_opt", "first"),
           epochs_sur=("epochs_sur", "first"),
           iterations=("iterations", "first"),
           k_batch=("k_batch", "first"),
           candidate_pool_M=("candidate_pool_M", "first"),
           gpu=("gpu", "first"),
           cuda_version=("cuda_version", "first"),
           python_version=("python_version", "first"),
       )
       .reset_index())

agg["runtime_mean_hms"] = pd.to_timedelta(agg["runtime_mean_sec"], unit="s").astype(str)
agg["runtime_std_hms"]  = pd.to_timedelta(agg["runtime_std_sec"], unit="s").astype(str)

# save for appendix
agg.to_csv("appendix_runtime_table.csv", index=False)
meta_df.to_csv("appendix_run_metadata.csv", index=False)

agg


,dataset,blackbox,strategy,n_runs,runtime_mean_sec,runtime_std_sec,batch_size,epochs_opt,epochs_sur,iterations,k_batch,candidate_pool_M,gpu,cuda_version,python_version,runtime_mean_hms,runtime_std_hms
0,bios,bios_csv,bo,16,27621.399975,12787.960299,256,3,4,75,16.0,1000.0,None,None,None,0 days 07:40:21.399975360,0 days 03:33:07.960298822
1,bios,bios_csv,disagreement,17,20616.651860,11458.231037,256,3,4,75,16.0,1000.0,None,None,None,0 days 05:43:36.651860050,0 days 03:10:58.231036540
2,jigsaw,hatebert,bo,90,16075.971184,17148.019113,256,3,4,100,8.0,1000.0,None,None,None,0 days 04:27:55.971183741,0 days 04:45:48.019112541
3,jigsaw,hatebert,bo_hybrid,2,2361.042044,257.473817,1024,3,4,75,32.0,1000.0,None,None,None,0 days 00:39:21.042043727,0 days 00:04:17.473817441
4,jigsaw,hatebert,disagreement,31,23744.058099,23148.487629,2056,3,4,75,8.0,1000.0,None,None,None,0 days 06:35:44.058099274,0 days 06:25:48.487629128


In [ ]:
import numpy as np
import pandas as pd

def _normalized_auec(df, t_col, err_col, t_max, prepend_t0=True, t0_value=0.0):
    d = df[[t_col, err_col]].dropna().copy()
    if d.empty:
        return np.nan
    d = d.sort_values(t_col).drop_duplicates(subset=[t_col], keep="last")
    d = d[d[t_col] <= t_max]
    if d.empty:
        return np.nan

    if prepend_t0 and float(d[t_col].iloc[0]) > t0_value:
        first_err = float(d[err_col].iloc[0])
        d = pd.concat([pd.DataFrame({t_col: [t0_value], err_col: [first_err]}), d], ignore_index=True)

    if float(d[t_col].iloc[-1]) < t_max:
        last_err = float(d[err_col].iloc[-1])
        d = pd.concat([d, pd.DataFrame({t_col: [t_max], err_col: [last_err]})], ignore_index=True)

    t = d[t_col].to_numpy(dtype=float)
    e = d[err_col].to_numpy(dtype=float)
    return float(np.trapz(e, t) / float(t_max))

def _interp_err_at_budget(df, t_col, err_col, budget):
    d = df[[t_col, err_col]].dropna().copy().sort_values(t_col)
    if d.empty:
        return np.nan

    if budget in d[t_col].values:
        return float(d.loc[d[t_col] == budget, err_col].iloc[-1])

    before = d[d[t_col] <= budget]
    after  = d[d[t_col] >= budget]
    if len(before) == 0 or len(after) == 0:
        return np.nan

    t1, e1 = float(before[t_col].iloc[-1]), float(before[err_col].iloc[-1])
    t2, e2 = float(after[t_col].iloc[0]),  float(after[err_col].iloc[0])
    if t1 == t2:
        return float(e1)
    return float(e1 + (e2 - e1) * (budget - t1) / (t2 - t1))

def summarize_by_strategy(
    ts: pd.DataFrame,
    eps_list=(0.02, 0.05),
    t_max=2000,
    budget=250,
    t_col="T_size",
    err_col="err_active",
    seed_col="seed",
    strategy_col="strategy",
) -> pd.DataFrame:
    """
    Baseline-equivalent summary, but pooled over hyperparams.
    Outputs one row per (strategy, epsilon) with:
      - queries_to_eps_mean/std/n_reached  (per-seed first-hit, then mean/std)
      - mean_crossing_T                   (population mean curve crosses eps)
      - auec_mean/std                     (per-seed)
      - error_at_budget_mean/std          (per-seed)
    """

    df = ts.copy()

    # numeric coercion
    for c in [t_col, err_col, seed_col]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=[t_col, err_col, seed_col, strategy_col])

    # IMPORTANT: if you have multiple runs for the same (strategy, seed),
    # keep them separate by run_id; otherwise you mix trajectories.
    # We'll treat each (strategy, seed, run_id) as a "seed-run".
    if "run_id" not in df.columns:
        df["run_id"] = "NA"

    # Deduplicate per timepoint within each run
    df = df.sort_values([strategy_col, seed_col, "run_id", t_col])
    df = df.drop_duplicates(subset=[strategy_col, seed_col, "run_id", t_col], keep="last")

    out_rows = []

    for strategy, sdf_all in df.groupby(strategy_col, dropna=False):
        # treat each (seed, run_id) as one replicate
        replicates = sdf_all[[seed_col, "run_id"]].drop_duplicates().values.tolist()

        # Per-replicate metrics
        auec_vals = []
        errB_vals = []
        first_hit = {eps: [] for eps in eps_list}

        for seed, run_id in replicates:
            rdf = sdf_all[(sdf_all[seed_col] == seed) & (sdf_all["run_id"] == run_id)].sort_values(t_col)

            auec_vals.append(_normalized_auec(rdf, t_col=t_col, err_col=err_col, t_max=t_max))
            errB_vals.append(_interp_err_at_budget(rdf, t_col=t_col, err_col=err_col, budget=budget))

            for eps in eps_list:
                hit = rdf[rdf[err_col] <= eps]
                first_hit[eps].append(float(hit[t_col].iloc[0]) if len(hit) else np.nan)

        # Population mean curve (mean over ALL replicates at each t)
        pop_curve = (
            sdf_all.groupby(t_col)[err_col]
                  .mean()
                  .reset_index()
                  .sort_values(t_col)
        )

        # Base row (shared across eps)
        base = {
            "strategy": strategy,
            "n_replicates": int(len(replicates)),
            "auec_mean": float(np.nanmean(auec_vals)) if np.any(~np.isnan(auec_vals)) else np.nan,
            "auec_std":  float(np.nanstd(auec_vals))  if np.any(~np.isnan(auec_vals)) else np.nan,
            f"error_at_{budget}_mean": float(np.nanmean(errB_vals)) if np.any(~np.isnan(errB_vals)) else np.nan,
            f"error_at_{budget}_std":  float(np.nanstd(errB_vals))  if np.any(~np.isnan(errB_vals)) else np.nan,
        }

        for eps in eps_list:
            vals = np.array(first_hit[eps], dtype=float)
            valid = vals[~np.isnan(vals)]

            crossed = pop_curve[pop_curve[err_col] <= eps]
            mean_crossing_T = float(crossed[t_col].min()) if len(crossed) else np.nan

            out_rows.append({
                **base,
                "epsilon": float(eps),
                "mean_crossing_T": mean_crossing_T,  # population-mean curve crossing
                f"queries_to_{eps}_mean": float(valid.mean()) if len(valid) else np.nan,
                f"queries_to_{eps}_std":  float(valid.std())  if len(valid) else np.nan,
                f"queries_to_{eps}_n_reached": int(len(valid)),
            })

    return pd.DataFrame(out_rows).sort_values(["strategy", "epsilon"]).reset_index(drop=True)


In [ ]:
# If you only want bo + disagreement:
ts_f = ts[ts["strategy"].isin(["bo", "disagreement"])].copy()

summary = summarize_by_strategy(
    ts_f,
    eps_list=(0.02, 0.05),
    t_max=2000,
    budget=250,
    t_col="T_size",
    err_col="err_active",
)

print(summary)


       strategy  n_replicates  auec_mean  auec_std  error_at_250_mean  \
0            bo            10   0.018182  0.006079           0.014639   
1            bo            10   0.018182  0.006079           0.014639   
2  disagreement             7   0.016648  0.005644           0.019280   
3  disagreement             7   0.016648  0.005644           0.019280   

   error_at_250_std  epsilon  mean_crossing_T  queries_to_0.02_mean  \
0          0.009681     0.02            164.0            124.400000   
1          0.009681     0.05            132.0                   NaN   
2          0.013369     0.02            148.0            122.857143   
3          0.013369     0.05             84.0                   NaN   

   queries_to_0.02_std  queries_to_0.02_n_reached  queries_to_0.05_mean  \
0            37.499867                       10.0                   NaN   
1                  NaN                        NaN                 100.4   
2            31.836316                        7.0    